# Notebook 4 | Statistical Independence

In [1]:
from foundations_of_probability_and_statistics.cars.car_distribution import create_car_distribution
from foundations_of_probability_and_statistics.cars.car_distribution import create_joint_car_distribution
from foundations_of_probability_and_statistics.cars.sample_from_car_distribution import sample_from_car_distribution

import pandas as pd

# show all rows of data frames and series per default
pd.set_option("display.max_rows", None)

## Independence and conditional independence

Generic random variables $X$ and $Y$ are independent when the joint factors for all values $x$ and $y$:

$$p(X = x, Y = y) = p(X = x) \, p(Y = y), \qquad p(x, y) = p(x) \, p(y).$$

Horsepower $H$ and color $C$ are not independent unconditionally, but they are conditionally independent
given the brand $B$: for every brand value $b$,

$$p(H = h, C = c \mid B = b) = p(H = h \mid B = b) \, p(C = c \mid B = b).$$

## Derivation

1. Take all $(h, c)$ pairs, e.g. $(300, \text{black})$.
2. Read $p(h \mid \text{Porsche}) = 0.4$ and $p(c \mid \text{Porsche}) = 0.4$ from the package.
3. Multiply: $p(300, \text{black} \mid \text{Porsche}) = 0.4 \cdot 0.4 = 0.16$.
4. The same entry sliced from the joint and divided by $p(\text{Porsche})$ agrees.
5. Without conditioning on the brand the factorization breaks, because the brand mixes the conditionals.

## Worked example (by hand)

Conditional independence within Porsche:

$$p(300, \text{black} \mid \text{Porsche}) = 0.4 \cdot 0.4 = 0.16.$$

Unconditional dependence of the same pair:

$$p(300, \text{black}) = 0.5 \cdot 0.1 \cdot 0.3 + 0.3 \cdot 0.4 \cdot 0.4 = 0.063 \neq p(300) \, p(\text{black}) = 0.17 \cdot 0.31 = 0.0527.$$

In [2]:
import math

joint = create_joint_car_distribution()
idx_slice = (slice(None), slice(None), slice(None))

# conditional independence: p(300, black | Porsche) = p(300 | Porsche) p(black | Porsche)
porsche_slice = joint.xs("Porsche", level="brand")
porsche_slice = porsche_slice / porsche_slice.sum()
assert math.isclose(porsche_slice.loc[(300, "black")], 0.4 * 0.4)
assert math.isclose(porsche_slice.loc[(300, "black")], 0.16)

# unconditional dependence: p(300, black) != p(300) p(black)
p_300_black = joint.xs(300, level="horsepower").xs("black", level="color").sum()
p_300 = joint.xs(300, level="horsepower").sum()
p_black = joint.xs("black", level="color").sum()
assert math.isclose(p_300_black, 0.5 * 0.1 * 0.3 + 0.3 * 0.4 * 0.4)
assert math.isclose(p_300_black, 0.063)
assert math.isclose(p_300 * p_black, 0.17 * 0.31, rel_tol=1e-9)
assert not math.isclose(p_300_black, p_300 * p_black)
p_300_black

np.float64(0.063)

## Generalization

One sample of 60,000 cars confirms both directions empirically: within each brand the empirical
$p(H = h, C = c \mid B = b)$ matches the product $p(H = h \mid B = b) \, p(C = c \mid B = b)$,
while the unconditional table does not factorize.

In [3]:
import numpy as np

cars = sample_from_car_distribution(n_cars=60_000, random_state=42)
car_distribution = create_car_distribution()

# per-brand empirical p(H = h, C = c | B = b) against the conditional product
max_gap = 0.0
for brand_code, brand_name in car_distribution.brand_names.items():
    sub = cars[cars["brand"] == brand_name]
    for h in car_distribution.horsepower_rvs[brand_code].xk:
        p_h = car_distribution.horsepower_rvs[brand_code].pmf(h)
        for c in car_distribution.color_rvs[brand_code].xk:
            color = car_distribution.color_names[c]
            p_c = car_distribution.color_rvs[brand_code].pmf(c)
            empirical = float(((sub["horsepower"] == h) & (sub["color"] == color)).mean())
            max_gap = max(max_gap, abs(empirical - p_h * p_c))
assert max_gap < 0.01
max_gap

np.float64(0.004100777271508982)

## References

- Blitzstein, J. K., Hwang, J. (2019): "Introduction to Probability", 2nd ed., Chapman & Hall/CRC, chapters "Conditional Probability" (independence of events) and "Joint Distributions", https://www.routledge.com/Introduction-to-Probability-Second-Edition/Blitzstein-Hwang/p/book/9781138369917 (free PDF: https://probabilitybook.net/).
- Wasserman, L. (2004): "All of Statistics: A Concise Course in Statistical Inference", Springer Texts in Statistics, chapter "Probability", https://doi.org/10.1007/978-0-387-21736-9.
- scipy.stats.rv_discrete, https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_discrete.html.